# 🏥 ClinicAI Clinical Inference & Explanations Pipeline
#### Production-grade Chest X-Ray diagnostic engine with GradCAM visual explanation, multi-scale TTA, clinical report card generator, per-class threshold grid search, and comprehensive performance metrics.

---

## 📋 Features & Capability Matrix
1. **Interactive Config Panel**: Set modes, target images, limits, and TTA parameters.
2. **Robust Multi-format Imaging**: PNG, JPEG, JPG, and DICOM (handling rescales, monochrome inversions, and missing metadata).
3. **Advanced Checkpoint Reconstructor**: Rebuilds the model backbone dynamically from checkpoint variables (supporting standard/EMA weights).
4. **GradCAM Visual Explanation**: Heatmaps overlaying pathological coordinates on target X-Rays.
5. **PDF Report Card Generator**: Compiles patient metadata, prediction tables, and GradCAM overlays into `report.pdf`.
6. **F1 Optimization Search**: Scans the `[0.05, 0.95]` space to find the optimal per-class probability decision boundaries.
7. **Test-Time Augmentation (TTA)**: Evaluates multiple scales `[288, 320, 352]` combined with horizontal flips.
8. **Clinical Error Analysis**: Renders grid plots of False Positives, False Negatives, Hardest Examples (highest loss), and True Positives.


## ⚙️ Interactive User Settings Panel
Modify the variables below to set the operation mode. Run the notebook top-to-bottom to execute prediction.


In [ ]:
# ==============================================================================
# CLINICAI PIPELINE INTERACTIVE SELECTOR
# ==============================================================================

# Operation Mode selector. Supported options:
#  - "single"  : Predictions, GradCAM explainability, and PDF report for one X-Ray.
#  - "batch"   : Processes a folder of images, outputs 'predictions.csv'.
#  - "dataset" : Evaluates the NIH test set, optimizes thresholds, saves plots/csvs.
MODE = "single"

# Mode 1: Single image inference settings
# Path to PNG, JPEG, JPG, or DICOM (.dcm) file
IMAGE_PATH = "/kaggle/input/nih-chest-xrays-data/images/00000003_000.png"

# Mode 2: GradCAM explainability threshold
# Create explainability overlays for classes whose predicted probability exceeds this threshold
GRADCAM_PROB_THRESHOLD = 0.40

# Mode 3: Batch prediction settings
# Directory containing images for batch prediction
BATCH_INPUT_FOLDER = "/kaggle/input/nih-chest-xrays-data/images/"
# Maximum number of images to predict (set to None to run all)
BATCH_LIMIT = 50

# Mode 9: Test-Time Augmentation (TTA)
# Toggle TTA (evaluates 3 scales [288, 320, 352] + horizontal flips)
USE_TTA = True


## 📦 Section 1 — Environment Setup & Package Installs
Installs missing packages (pydicom, reportlab, grad-cam) and imports libraries.


In [ ]:
import subprocess, sys

def pip_install(*packages):
    """Install python packages silently. Skips if already present."""
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", *packages],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
    except Exception as e:
        print(f"Warning: Failed to install package(s) {packages}: {e}")

# Run dynamic installations for Kaggle kernel environment
print("Installing dependencies...")
pip_install("timm>=0.9.12", "albumentations>=1.3.1", "grad-cam>=1.4.8", "reportlab>=4.0.0", "pydicom>=2.4.0")

# Standard imports
import os, gc, math, time, json, random, warnings, copy
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from typing import Dict, List, Optional, Tuple, Union

# Core DS
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg") # Headless plot saving
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# SciKit-Learn evaluation
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split

# Pytorch Grad-CAM
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    GRADCAM_AVAILABLE = True
except ImportError:
    GRADCAM_AVAILABLE = False
    print("WARNING: pytorch_grad_cam library not loaded.")

# Seed everything for deterministic plots/splitting
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

# Configure run device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inference Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Configure plotting styles
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 15,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.titlesize': 17
})

warnings.filterwarnings("ignore")
print("Environment and packages loaded successfully.")


## 🧠 Section 2 — Model Reconstructor & Checkpoint Parser
Loads `best_model.pth` and dynamically updates parameters, rebuilding the backbone correctly without hardcoded dimensions.


In [ ]:
class Config:
    """
    Single source of truth parameters.
    Updated dynamically on loading best_model.pth checkpoint.
    """
    DATA_DIR = Path("/kaggle/input/nih-chest-xrays-data")
    IMAGE_DIR = DATA_DIR / "images"
    META_CSV = DATA_DIR / "Data_Entry_2017_v2020.csv"
    OUTPUT_DIR = Path("/kaggle/working")
    CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
    GRAD_CAM_DIR = OUTPUT_DIR / "gradcam"
    
    CLASSES = [
        "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
        "Mass", "Nodule", "Pneumonia", "Pneumothorax",
        "Consolidation", "Edema", "Emphysema", "Fibrosis",
        "Pleural_Thickening", "Hernia",
    ]
    NUM_CLASSES = len(CLASSES)
    IMG_SIZE = 320
    MEAN = [0.485, 0.456, 0.406]
    STD = [0.229, 0.224, 0.225]
    BACKBONE = "convnext_base"
    PRETRAINED = False
    DROP_RATE = 0.2
    DROP_PATH_RATE = 0.1
    AMP = True
    TTA_SCALES = [288, 320, 352]
    SEED = 42

CFG = Config()
CFG.GRAD_CAM_DIR.mkdir(parents=True, exist_ok=True)

class ChestXrayModel(nn.Module):
    """
    Multi-label classification head built on top of a specified timm backbone.
    """
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        # Recreate exact backbone (no pretrained downloads required during checkpoint loading)
        self.backbone = timm.create_model(
            cfg.BACKBONE,
            pretrained=False,
            num_classes=0,
            global_pool="avg",
            drop_rate=cfg.DROP_RATE,
            drop_path_rate=cfg.DROP_PATH_RATE,
        )
        num_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(num_features),
            nn.Dropout(p=cfg.DROP_RATE),
            nn.Linear(num_features, 512),
            nn.GELU(),
            nn.Dropout(p=cfg.DROP_RATE / 2),
            nn.Linear(512, cfg.NUM_CLASSES),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        logits = self.head(features)
        return logits

    def get_gradcam_layer(self) -> nn.Module:
        """Selects the final convolutional layer suitable for explainable heatmap generation."""
        backbone_name = self.cfg.BACKBONE.lower()
        if "convnext" in backbone_name:
            return self.backbone.stages[-1].blocks[-1]
        elif "densenet" in backbone_name:
            return self.backbone.features.denseblock4
        elif "efficientnet" in backbone_name:
            return self.backbone.blocks[-1]
        elif "swin" in backbone_name:
            return self.backbone.layers[-1].blocks[-1]
        else:
            # Suffix/Weight search fallback
            for module in reversed(list(self.backbone.modules())):
                if hasattr(module, "weight"):
                    return module
            return self.backbone

def locate_checkpoint() -> Path:
    """Searches for best_model.pth in standard Kaggle/local folders."""
    possible_paths = [
        Path("/kaggle/input/checkpoints/best_model.pth"),
        Path("/kaggle/working/checkpoints/best_model.pth"),
        Path("checkpoints/best_model.pth"),
        Path("best_model.pth"),
    ]
    for p in possible_paths:
        if p.exists():
            return p
    # Fallback to search in working directory
    ckpt_files = list(Path(".").glob("**/best_model.pth"))
    if ckpt_files:
        return ckpt_files[0]
    raise FileNotFoundError("Could not locate best_model.pth. Place checkpoint in workspace.")

def load_checkpoint_and_rebuild(device: torch.device) -> Tuple[ChestXrayModel, Dict]:
    """
    Loads best_model.pth, overrides Config properties, instantiates ChestXrayModel,
    detects model vs ema state_dict, and maps key structures successfully.
    """
    ckpt_path = locate_checkpoint()
    print(f"Loading checkpoint from: {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location=device)
    
    # Overwrite configuration parameters if stored in checkpoint
    if "cfg" in checkpoint:
        print("Restoring Config settings from checkpoint...")
        cfg_dict = checkpoint["cfg"]
        for k, v in cfg_dict.items():
            if hasattr(CFG, k):
                # Ensure directory structures are parsed as Paths
                if k.endswith("_DIR") or k.endswith("_FILE") or k == "DATA_DIR" or k == "OUTPUT_DIR":
                    setattr(CFG, k, Path(v))
                else:
                    setattr(CFG, k, v)
                print(f"  {k} = {v}")
                
    # Instantiate Model with rebuilt configuration
    model = ChestXrayModel(CFG).to(device)
    
    # Detect state dict formats (checking ema vs model weights keys)
    state_dict = None
    if "ema" in checkpoint:
        state_dict = checkpoint["ema"]
        print("Loading EMA copy from checkpoint (better validation generalization).")
    elif "model" in checkpoint:
        state_dict = checkpoint["model"]
        print("Loading standard model state_dict weights.")
    else:
        state_dict = checkpoint
        print("Using checkpoint dictionary as raw state_dict.")
        
    # Clean possible DDP/Wrapper prefixes from state dict keys
    model_state = model.state_dict()
    cleaned_state = {}
    for k, v in state_dict.items():
        name = k
        if name.startswith("ema_model."):
            name = name[10:]
        elif name.startswith("model."):
            name = name[6:]
        elif name.startswith("module."):
            name = name[7:]
            
        if name in model_state:
            cleaned_state[name] = v
        else:
            # Handle possible key naming discrepancies by matching matching suffixes
            found = False
            for mk in model_state.keys():
                if mk.endswith(name) or name.endswith(mk):
                    cleaned_state[mk] = v
                    found = True
                    break
            if not found:
                print(f"  Warning: Checkpoint key '{k}' could not be matched to model state.")
                
    model.load_state_dict(cleaned_state, strict=False)
    model.eval()
    print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f} M")
    return model, checkpoint


## 📷 Section 3 — Image Reader & Dataset Pipeline
Reads PNG, JPEG, JPG, and DICOM formats. Applies rescalings, photometric MONOCHROME1 inversions, and normalizations.


In [ ]:
def load_image_as_rgb(image_path: Union[str, Path]) -> np.ndarray:
    """
    Loads medical image files (PNG, JPEG, JPG, or DICOM .dcm) and returns a 3-channel RGB numpy array.
    Handles contrast rescales and monochromatic color spaces.
    """
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Imaging file not found at: {image_path}")
        
    ext = path.suffix.lower()
    
    if ext == ".dcm":
        try:
            import pydicom
        except ImportError:
            raise ImportError("pydicom is required to parse DICOM datasets. Run pip install pydicom.")
            
        ds = pydicom.dcmread(str(path))
        pixel_array = ds.pixel_array
        
        # Apply Rescale Slope and Intercept if specified in metadata
        slope = getattr(ds, "RescaleSlope", 1.0)
        intercept = getattr(ds, "RescaleIntercept", 0.0)
        if slope != 1.0 or intercept != 0.0:
            pixel_array = pixel_array.astype(np.float32) * slope + intercept
            
        # Rescale values to 8-bit [0, 255] window range
        pixel_array = pixel_array.astype(np.float32)
        p_min, p_max = pixel_array.min(), pixel_array.max()
        if p_max > p_min:
            pixel_array = (pixel_array - p_min) / (p_max - p_min) * 255.0
        else:
            pixel_array = np.zeros_like(pixel_array)
        img = pixel_array.astype(np.uint8)
        
        # Invert intensity values if Photometric Interpretation is MONOCHROME1
        photometric = getattr(ds, "PhotometricInterpretation", "MONOCHROME2")
        if photometric == "MONOCHROME1":
            img = 255 - img
    else:
        # Load standard PNG, JPEG, JPG using cv2
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            # Fallback to PIL in case of loading hiccups
            try:
                with Image.open(path) as pil_img:
                    img = np.array(pil_img.convert("L"))
            except Exception as e:
                raise IOError(f"Failed to read image {image_path}: {e}")
                
    # Force expand to 3-channel RGB for ImageNet backbone compatibility
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif len(img.shape) == 3 and img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif len(img.shape) == 3 and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    elif len(img.shape) == 3 and img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
    return img

def build_transforms(mode: str, cfg: Config) -> A.Compose:
    """Constructs standard validation / test transforms."""
    if mode in ("val", "test"):
        return A.Compose([
            A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE, always_apply=True),
            A.Normalize(mean=cfg.MEAN, std=cfg.STD, always_apply=True),
            ToTensorV2(always_apply=True),
        ])
    else:
        raise ValueError(f"Unknown transform mode: {mode}")

def build_tta_transforms(scale: int, cfg: Config) -> A.Compose:
    """Constructs scale-specific transforms for TTA."""
    return A.Compose([
        A.Resize(scale, scale, always_apply=True),
        A.Normalize(mean=cfg.MEAN, std=cfg.STD, always_apply=True),
        ToTensorV2(always_apply=True),
    ])

class ChestXrayInferenceDataset(Dataset):
    """
    General dataset for batch predictions.
    Gracefully handles corrupted image data by outputting a black canvas fallback.
    """
    def __init__(self, df: pd.DataFrame, image_dir: Path, transform: A.Compose, cfg: Config):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.cfg = cfg
        # Map labels if they are found in CSV; otherwise default to zeroes
        self.has_labels = all(cls in df.columns for cls in cfg.CLASSES)
        if self.has_labels:
            self.labels = df[cfg.CLASSES].values.astype(np.float32)
        else:
            self.labels = np.zeros((len(df), cfg.NUM_CLASSES), dtype=np.float32)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, str]:
        row = self.df.iloc[idx]
        img_name = row["Image Index"]
        img_path = self.image_dir / img_name
        
        try:
            img = load_image_as_rgb(img_path)
        except Exception as e:
            # Fallback canvas to avoid crashing dataset evaluations
            print(f"WARNING: Image loading failure on '{img_name}': {e}. Loading black canvas placeholder.")
            img = np.zeros((self.cfg.IMG_SIZE, self.cfg.IMG_SIZE, 3), dtype=np.uint8)
            
        augmented = self.transform(image=img)
        image = augmented["image"]
        label = torch.from_numpy(self.labels[idx])
        return image, label, img_name


## 🗺️ Section 4 — Explainability (GradCAM)
Highlights pathology coordinates using final conv layers. Generates visual grids: Original, Heatmap, Overlay.


In [ ]:
def generate_gradcam_visualizations(
    model: nn.Module,
    image_path: Union[str, Path],
    class_indices: List[int],
    cfg: Config,
    device: torch.device,
    save_dir: Optional[Path] = None,
    show_plot: bool = True
) -> List[Path]:
    """
    Generates side-by-side diagnostic figures for active classes.
    Layout: Original Image | GradCAM Heatmap | Overlaid coordinates.
    """
    if not GRADCAM_AVAILABLE:
        print("WARNING: pytorch_grad_cam is not installed. Skipping overlays.")
        return []
        
    save_dir = save_dir or cfg.GRAD_CAM_DIR
    save_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        img_rgb = load_image_as_rgb(image_path)
    except Exception as e:
        print(f"Skipping GradCAM. Could not load image {image_path}: {e}")
        return []
        
    img_resized = cv2.resize(img_rgb, (cfg.IMG_SIZE, cfg.IMG_SIZE))
    img_normalized = img_resized.astype(np.float32) / 255.0
    
    tfm = build_transforms("val", cfg)
    tensor = tfm(image=img_rgb)["image"].unsqueeze(0).to(device)
    
    target_layer = model.get_gradcam_layer()
    cam = GradCAM(model=model, target_layers=[target_layer])
    
    model.eval()
    with torch.no_grad():
        with autocast(enabled=cfg.AMP):
            logits = model(tensor)
    probs = torch.sigmoid(logits).cpu().numpy().squeeze()
    
    saved_paths = []
    for cls_idx in class_indices:
        cls_name = cfg.CLASSES[cls_idx]
        prob = probs[cls_idx]
        
        # Calculate coordinate heatmap
        targets = [ClassifierOutputTarget(cls_idx)]
        cam_map = cam(input_tensor=tensor, targets=targets)[0]
        
        # Superimpose overlay
        overlay = show_cam_on_image(img_normalized, cam_map, use_rgb=True)
        
        # Renders the side-by-side explanation grid
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(img_resized)
        axes[0].set_title("Original Patient X-Ray", fontsize=11, fontweight="bold")
        axes[0].axis("off")
        
        axes[1].imshow(cam_map, cmap="jet")
        axes[1].set_title(f"{cls_name} Coordinate Map", fontsize=11, fontweight="bold")
        axes[1].axis("off")
        
        axes[2].imshow(overlay)
        axes[2].set_title(f"Superimposed Overlay (p={prob*100:.1f}%)", fontsize=11, fontweight="bold")
        axes[2].axis("off")
        
        plt.suptitle(f"GradCAM Explainability Analysis — Pathology: {cls_name}", fontsize=14, fontweight="bold")
        plt.tight_layout()
        
        img_name = Path(image_path).stem
        save_path = save_dir / f"gradcam_{img_name}_{cls_name.lower()}.png"
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        if show_plot:
            plt.show()
        plt.close()
        
        saved_paths.append(save_path)
        print(f"  GradCAM visualization exported for {cls_name} (p={prob:.3f}) at: {save_path}")
        
    return saved_paths


## 📄 Section 5 — Clinical PDF Report Generator
Compiles patient metadata, probabilities, and GradCAM overlays into a ReportLab `report.pdf` card.


In [ ]:
def generate_pdf_report(
    output_path: Union[str, Path],
    patient_img_path: Union[str, Path],
    gradcam_img_path: Optional[Union[str, Path]],
    predictions: Dict[str, float],
    thresholds: Dict[str, float],
    top_diagnosis: str,
    top_prob: float,
    model_name: str,
    inference_time: float,
    device_name: str
) -> None:
    """
    Generates a publication-quality clinical PDF report using ReportLab.
    """
    try:
        from reportlab.lib.pagesizes import letter
        from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib import colors
    except ImportError:
        print("reportlab is not installed. Skipping PDF report generation.")
        return

    doc = SimpleDocTemplate(
        str(output_path),
        pagesize=letter,
        leftMargin=36,
        rightMargin=36,
        topMargin=36,
        bottomMargin=36
    )

    styles = getSampleStyleSheet()
    
    # Custom report styles
    title_style = ParagraphStyle(
        'DocTitle',
        parent=styles['Heading1'],
        fontName='Helvetica-Bold',
        fontSize=24,
        leading=28,
        textColor=colors.HexColor('#1B365D'),
        spaceAfter=15
    )
    
    header_style = ParagraphStyle(
        'SectionHeader',
        parent=styles['Heading2'],
        fontName='Helvetica-Bold',
        fontSize=14,
        leading=18,
        textColor=colors.HexColor('#1B365D'),
        spaceBefore=10,
        spaceAfter=6,
        keepWithNext=True
    )
    
    body_style = ParagraphStyle(
        'BodyDark',
        parent=styles['Normal'],
        fontName='Helvetica',
        fontSize=10,
        leading=14,
        textColor=colors.HexColor('#2C3E50')
    )
    
    body_bold = ParagraphStyle(
        'BodyDarkBold',
        parent=body_style,
        fontName='Helvetica-Bold'
    )
    
    meta_style = ParagraphStyle(
        'Metadata',
        parent=styles['Normal'],
        fontName='Helvetica-Oblique',
        fontSize=9,
        textColor=colors.HexColor('#7F8C8D')
    )

    elements = []

    # Header Banner
    title_p = Paragraph("🏥 ClinicAI Clinical Diagnostic Report", title_style)
    elements.append(title_p)
    
    # Divider Rule
    line_table = Table([[""]], colWidths=[540])
    line_table.setStyle(TableStyle([
        ('LINEBELOW', (0,0), (-1,-1), 2, colors.HexColor('#1B365D')),
        ('BOTTOMPADDING', (0,0), (-1,-1), 0),
        ('TOPPADDING', (0,0), (-1,-1), 0),
    ]))
    elements.append(line_table)
    elements.append(Spacer(1, 10))

    # Metadata Grid
    meta_data = [
        [Paragraph("<b>Report Date:</b>", body_bold), Paragraph(time.strftime("%Y-%m-%d %H:%M:%S"), body_style),
         Paragraph("<b>Model Architecture:</b>", body_bold), Paragraph(model_name, body_style)],
        [Paragraph("<b>Inference Device:</b>", body_bold), Paragraph(device_name, body_style),
         Paragraph("<b>Inference Time:</b>", body_bold), Paragraph(f"{inference_time:.3f} seconds", body_style)],
        [Paragraph("<b>Top Diagnosis:</b>", body_bold), Paragraph(f"<b>{top_diagnosis}</b> ({top_prob*100:.1f}%)", body_bold),
         Paragraph("<b>Input Image:</b>", body_bold), Paragraph(Path(patient_img_path).name, body_style)]
    ]
    meta_table = Table(meta_data, colWidths=[110, 160, 120, 150])
    meta_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#F8F9FA')),
        ('ALIGN', (0,0), (-1,-1), 'LEFT'),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('BOTTOMPADDING', (0,0), (-1,-1), 6),
        ('TOPPADDING', (0,0), (-1,-1), 6),
        ('LEFTPADDING', (0,0), (-1,-1), 8),
        ('RIGHTPADDING', (0,0), (-1,-1), 8),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#BDC3C7')),
    ]))
    elements.append(meta_table)
    elements.append(Spacer(1, 15))

    # Imaging Finding block
    elements.append(Paragraph("📷 Imaging Findings", header_style))
    
    img_elements = []
    
    # Rescale functions to format images inside the letter dimensions safely
    def get_resized_image(path, target_width):
        try:
            im = cv2.imread(str(path))
            h, w = im.shape[:2]
            aspect = h / w
            target_height = int(target_width * aspect)
            temp_path = Path(f"temp_pdf_{Path(path).stem}_{target_width}.png")
            cv2.imwrite(str(temp_path), cv2.resize(im, (target_width, target_height)))
            return str(temp_path), target_width, target_height
        except Exception as e:
            print(f"Error scaling image: {e}")
            return str(path), target_width, target_width
            
    temp_files = []
    
    if gradcam_img_path and Path(gradcam_img_path).exists():
        p1, w1, h1 = get_resized_image(patient_img_path, 250)
        p2, w2, h2 = get_resized_image(gradcam_img_path, 250)
        temp_files.extend([p1, p2])
        
        row_images = [
            RLImage(p1, width=250, height=h1),
            RLImage(p2, width=250, height=h2)
        ]
        img_table = Table([row_images], colWidths=[270, 270])
        img_table.setStyle(TableStyle([
            ('ALIGN', (0,0), (-1,-1), 'CENTER'),
            ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
            ('LEFTPADDING', (0,0), (-1,-1), 0),
            ('RIGHTPADDING', (0,0), (-1,-1), 0),
        ]))
        img_elements.append(img_table)
    else:
        p1, w1, h1 = get_resized_image(patient_img_path, 350)
        temp_files.append(p1)
        img_elements.append(RLImage(p1, width=350, height=h1))
        
    elements.extend(img_elements)
    elements.append(Spacer(1, 15))

    # Diagnostic Findings block
    elements.append(Paragraph("📊 Diagnostic Findings & Probability Chart", header_style))
    
    findings_data = [
        [Paragraph("<b>Thoracic Pathology</b>", body_bold), 
         Paragraph("<b>Probability</b>", body_bold), 
         Paragraph("<b>Class Decision Boundary</b>", body_bold), 
         Paragraph("<b>Clinical Classification</b>", body_bold)]
    ]
    
    sorted_preds = sorted(predictions.items(), key=lambda x: x[1], reverse=True)
    
    for cls, prob in sorted_preds:
        thresh = thresholds.get(cls, 0.5)
        is_pos = prob >= thresh
        
        class_text = f"<font color='#C0392B'><b>POSITIVE</b></font>" if is_pos else f"<font color='#27AE60'>NEGATIVE</font>"
        prob_text = f"<b>{prob*100:.2f}%</b>" if is_pos else f"{prob*100:.2f}%"
        
        findings_data.append([
            Paragraph(cls, body_bold if is_pos else body_style),
            Paragraph(prob_text, body_style),
            Paragraph(f"{thresh*100:.1f}%", body_style),
            Paragraph(class_text, body_style)
        ])
        
    findings_table = Table(findings_data, colWidths=[180, 110, 110, 140])
    findings_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#EBF5FB')),
        ('ALIGN', (0,0), (-1,-1), 'LEFT'),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('BOTTOMPADDING', (0,0), (-1,-1), 4),
        ('TOPPADDING', (0,0), (-1,-1), 4),
        ('LEFTPADDING', (0,0), (-1,-1), 8),
        ('RIGHTPADDING', (0,0), (-1,-1), 8),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#F4F6F7')]),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#BDC3C7')),
    ]))
    elements.append(findings_table)
    elements.append(Spacer(1, 20))

    # Footnotes
    disclaimer_text = (
        "<b>Clinical Disclaimer:</b> This report is generated by an artificial intelligence model (ClinicAI) "
        "trained on the NIH ChestX-ray14 dataset. It is intended for research and educational purposes only. "
        "All model recommendations must be verified by a board-certified radiologist prior to any clinical intervention or treatment."
    )
    elements.append(Paragraph(disclaimer_text, meta_style))

    doc.build(elements)
    print(f"Clinical PDF Report successfully generated at: {output_path}")
    
    # Clean temporary rescaled images
    for tf in temp_files:
        try:
            if os.path.exists(tf):
                os.remove(tf)
        except Exception:
            pass


## 📈 Section 6 — Metrics and Evaluation Graphs
Computes standard multilabel classification scores and exports high-quality ROC, PR, Confusion, and Calibration figures.


In [ ]:
def compute_evaluation_metrics(y_true: np.ndarray, y_prob: np.ndarray, thresholds: np.ndarray) -> Dict:
    """
    Computes all standard multi-label metrics: Accuracy, Precision, Recall, Specificity, F1, ROC-AUC, AP.
    Returns Macro, Micro, and Weighted metrics alongside per-class dictionaries.
    """
    y_pred = (y_prob >= thresholds).astype(int)
    num_classes = y_true.shape[1]
    
    metrics = {}
    
    # Multi-label ROC-AUC calculations
    try:
        metrics["auc_macro"] = roc_auc_score(y_true, y_prob, average="macro")
        metrics["auc_micro"] = roc_auc_score(y_true.ravel(), y_prob.ravel())
        metrics["auc_weighted"] = roc_auc_score(y_true, y_prob, average="weighted")
    except Exception:
        metrics["auc_macro"] = metrics["auc_micro"] = metrics["auc_weighted"] = 0.5
        
    # Multi-label Average Precision scores
    try:
        metrics["ap_macro"] = average_precision_score(y_true, y_prob, average="macro")
        metrics["ap_micro"] = average_precision_score(y_true.ravel(), y_prob.ravel())
        metrics["ap_weighted"] = average_precision_score(y_true, y_prob, average="weighted")
    except Exception:
        metrics["ap_macro"] = metrics["ap_micro"] = metrics["ap_weighted"] = 0.0
        
    # F1, Precision, Recall averages
    metrics["f1_macro"] = f1_score(y_true, y_pred, average="macro", zero_division=0)
    metrics["f1_micro"] = f1_score(y_true, y_pred, average="micro", zero_division=0)
    metrics["f1_weighted"] = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    
    metrics["precision_macro"] = precision_score(y_true, y_pred, average="macro", zero_division=0)
    metrics["precision_micro"] = precision_score(y_true, y_pred, average="micro", zero_division=0)
    metrics["precision_weighted"] = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    
    metrics["recall_macro"] = recall_score(y_true, y_pred, average="macro", zero_division=0)
    metrics["recall_micro"] = recall_score(y_true, y_pred, average="micro", zero_division=0)
    metrics["recall_weighted"] = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    
    metrics["accuracy_macro"] = (y_true == y_pred).mean()
    
    # Specificity calculation per class
    specificities = []
    for i in range(num_classes):
        cm = confusion_matrix(y_true[:, i], y_pred[:, i])
        tn = cm[0, 0] if cm.shape == (2, 2) else len(y_true[:, i]) - y_true[:, i].sum()
        fp = cm[0, 1] if cm.shape == (2, 2) else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 1.0
        specificities.append(spec)
    metrics["specificity_macro"] = np.mean(specificities)
    
    # Populate per-class specifics
    per_class = {}
    for i, cls in enumerate(CFG.CLASSES):
        n_pos = y_true[:, i].sum()
        cls_auc = roc_auc_score(y_true[:, i], y_prob[:, i]) if 0 < n_pos < len(y_true) else 0.5
        cls_ap = average_precision_score(y_true[:, i], y_prob[:, i]) if 0 < n_pos < len(y_true) else 0.0
        cls_p = precision_score(y_true[:, i], y_pred[:, i], zero_division=0)
        cls_r = recall_score(y_true[:, i], y_pred[:, i], zero_division=0)
        cls_f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
        
        per_class[cls] = {
            "auc": cls_auc,
            "ap": cls_ap,
            "precision": cls_p,
            "recall": cls_r,
            "f1": cls_f1,
            "specificity": specificities[i],
            "prevalence": n_pos / len(y_true)
        }
    metrics["per_class"] = per_class
    
    return metrics

def plot_performance_graphs(y_true: np.ndarray, y_prob: np.ndarray, thresholds: np.ndarray, save_dir: Path) -> None:
    """
    Generates and saves publication-quality evaluation figures.
    Outputs: roc_curves.png, pr_curves.png, confusion_matrix.png,
             calibration_curves.png, per_class_auroc.png, probability_distributions.png.
    """
    save_dir.mkdir(parents=True, exist_ok=True)
    y_pred = (y_prob >= thresholds).astype(int)
    
    # 1. ROC Curves (2x7 Grid)
    fig, axes = plt.subplots(2, 7, figsize=(26, 8))
    axes = axes.flatten()
    for i, cls in enumerate(CFG.CLASSES):
        n_pos = y_true[:, i].sum()
        if n_pos == 0:
            axes[i].axis("off")
            continue
        fpr, tpr, _ = roc_curve(y_true[:, i], y_prob[:, i])
        auc_val = roc_auc_score(y_true[:, i], y_prob[:, i])
        axes[i].plot(fpr, tpr, color="#E74C3C", lw=2, label=f"AUC={auc_val:.3f}")
        axes[i].plot([0, 1], [0, 1], "k--", lw=1)
        axes[i].set_title(cls, fontsize=9, fontweight="bold")
        axes[i].set_xlabel("FPR", fontsize=8)
        axes[i].set_ylabel("TPR", fontsize=8)
        axes[i].legend(fontsize=8, loc="lower right")
        axes[i].set_xlim([0, 1])
        axes[i].set_ylim([0, 1])
    plt.suptitle("Per-Class Receiver Operating Characteristic (ROC) Curves", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_dir / "roc_curves.png", dpi=150)
    plt.close()
    
    # 2. PR Curves (2x7 Grid)
    fig, axes = plt.subplots(2, 7, figsize=(26, 8))
    axes = axes.flatten()
    for i, cls in enumerate(CFG.CLASSES):
        n_pos = y_true[:, i].sum()
        if n_pos == 0:
            axes[i].axis("off")
            continue
        prec, rec, _ = precision_recall_curve(y_true[:, i], y_prob[:, i])
        ap_val = average_precision_score(y_true[:, i], y_prob[:, i])
        axes[i].plot(rec, prec, color="#2980B9", lw=2, label=f"AP={ap_val:.3f}")
        axes[i].set_title(cls, fontsize=9, fontweight="bold")
        axes[i].set_xlabel("Recall", fontsize=8)
        axes[i].set_ylabel("Precision", fontsize=8)
        axes[i].legend(fontsize=8, loc="lower left")
        axes[i].set_xlim([0, 1])
        axes[i].set_ylim([0, 1])
    plt.suptitle("Per-Class Precision-Recall (PR) Curves", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_dir / "pr_curves.png", dpi=150)
    plt.close()
    
    # 3. Confusion Matrices (2x7 Grid)
    fig, axes = plt.subplots(2, 7, figsize=(26, 8))
    axes = axes.flatten()
    for i, cls in enumerate(CFG.CLASSES):
        cm = confusion_matrix(y_true[:, i], y_pred[:, i])
        if cm.shape != (2, 2):
            axes[i].axis("off")
            continue
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i], cbar=False,
                    xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"], annot_kws={"size": 10})
        axes[i].set_title(cls, fontsize=9, fontweight="bold")
        axes[i].set_ylabel("True", fontsize=8)
        axes[i].set_xlabel("Pred", fontsize=8)
    plt.suptitle("Per-Class Confusion Matrices", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_dir / "confusion_matrix.png", dpi=150)
    plt.close()
    
    # 4. Calibration Curves (2x7 Grid)
    fig, axes = plt.subplots(2, 7, figsize=(26, 8))
    axes = axes.flatten()
    for i, cls in enumerate(CFG.CLASSES):
        n_pos = y_true[:, i].sum()
        if n_pos < 10:
            axes[i].axis("off")
            continue
        prob_true, prob_pred = calibration_curve(y_true[:, i], y_prob[:, i], n_bins=10)
        axes[i].plot(prob_pred, prob_true, marker='o', linewidth=2, color='#27AE60', label='Model')
        axes[i].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Ideal')
        axes[i].set_title(cls, fontsize=9, fontweight='bold')
        axes[i].set_xlabel('Pred Prob', fontsize=8)
        axes[i].set_ylabel('True Prob', fontsize=8)
        axes[i].legend(fontsize=8)
    plt.suptitle("Per-Class Calibration Curves (Reliability Diagrams)", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_dir / "calibration_curves.png", dpi=150)
    plt.close()
    
    # 5. Per-class AUROC Bar Chart
    aucs = []
    classes = []
    for i, cls in enumerate(CFG.CLASSES):
        if y_true[:, i].sum() > 0:
            aucs.append(roc_auc_score(y_true[:, i], y_prob[:, i]))
            classes.append(cls)
            
    idx = np.argsort(aucs)[::-1]
    sorted_classes = np.array(classes)[idx]
    sorted_aucs = np.array(aucs)[idx]
    
    plt.figure(figsize=(14, 6))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(sorted_aucs)))
    bars = plt.bar(sorted_classes, sorted_aucs, color=colors, edgecolor='gray')
    mean_auc = np.mean(aucs)
    plt.axhline(mean_auc, color="navy", linestyle="--", lw=2, label=f"Macro AUROC = {mean_auc:.4f}")
    plt.ylim([0.45, 1.0])
    plt.ylabel("ROC-AUC Score", fontsize=12)
    plt.title("Per-Class ROC-AUC Performance (Sorted Descending)", fontsize=16, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=10)
    plt.legend(fontsize=12, loc="upper right")
    for bar, val in zip(bars, sorted_aucs):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_dir / "per_class_auroc.png", dpi=150)
    plt.close()
    
    # 6. Histograms
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    axes[0].hist(y_prob.ravel(), bins=50, color="#8E44AD", edgecolor="white", alpha=0.7)
    axes[0].set_yscale("log")
    axes[0].set_title("Distribution of All Predicted Probabilities", fontsize=14, fontweight="bold")
    axes[0].set_xlabel("Probability", fontsize=12)
    axes[0].set_ylabel("Count (Log Scale)", fontsize=12)
    
    axes[1].hist(thresholds, bins=15, color="#D35400", edgecolor="white", alpha=0.7)
    axes[1].axvline(0.5, color="black", linestyle="--", label="Standard 0.5 Threshold")
    axes[1].set_title("Distribution of Optimized Class Thresholds", fontsize=14, fontweight="bold")
    axes[1].set_xlabel("Optimized Threshold Value", fontsize=12)
    axes[1].set_ylabel("Number of Classes", fontsize=12)
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(save_dir / "probability_distributions.png", dpi=150)
    plt.close()
    
    print("All evaluation plots generated successfully in output directory.")


## ⚙️ Section 7 — Threshold Optimization & Error Analysis
Searches `[0.05, 0.95]` for F1 optimization. Displays False Positives, False Negatives, and Hardest Cases for error audits.


In [ ]:
def run_threshold_optimization(y_true: np.ndarray, y_prob: np.ndarray) -> np.ndarray:
    """
    Searches the threshold space [0.05, 0.95] (step 0.01) to maximize the F1-score for each class.
    Saves thresholds to thresholds.json.
    """
    optimized_thresholds = np.zeros(CFG.NUM_CLASSES)
    thresh_range = np.linspace(0.05, 0.95, 91)
    
    print("\nStarting Threshold Optimization Grid Search (0.05 to 0.95)...")
    print(f"{'Class':<22} | {'Default F1 (0.5)':<16} | {'Best F1':<8} | {'Best Threshold':<14}")
    print("-" * 75)
    
    thresh_dict = {}
    for i, cls in enumerate(CFG.CLASSES):
        n_pos = y_true[:, i].sum()
        if n_pos == 0 or n_pos == len(y_true):
            optimized_thresholds[i] = 0.5
            thresh_dict[cls] = 0.5
            continue
            
        f1_at_05 = f1_score(y_true[:, i], (y_prob[:, i] >= 0.5).astype(int), zero_division=0)
        
        best_f1 = -1.0
        best_t = 0.5
        for t in thresh_range:
            score = f1_score(y_true[:, i], (y_prob[:, i] >= t).astype(int), zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_t = t
                
        optimized_thresholds[i] = best_t
        thresh_dict[cls] = float(best_t)
        print(f"{cls:<22} | {f1_at_05:<16.4f} | {best_f1:<8.4f} | {best_t:<14.2f}")
        
    with open("thresholds.json", "w") as f:
        json.dump(thresh_dict, f, indent=4)
        
    print("\nOptimized thresholds saved to: thresholds.json")
    return optimized_thresholds

def run_error_analysis(df: pd.DataFrame, y_true: np.ndarray, y_prob: np.ndarray, thresholds: np.ndarray, image_dir: Path, n: int = 4) -> None:
    """
    Performs clinical error audit: False Positives, False Negatives, Hardest Cases (highest loss), and True Positives.
    Saves grids as local PNGs.
    """
    y_pred = (y_prob >= thresholds).astype(int)
    
    print("\n" + "="*50)
    print("  RUNNING CLINICAL ERROR AUDIT  ")
    print("="*50)
    
    # We audit up to 3 classes to prevent rendering too many charts
    classes_to_audit = CFG.CLASSES[:3]
    
    for i, cls in enumerate(CFG.CLASSES):
        if cls not in classes_to_audit:
            continue
            
        gt = y_true[:, i]
        prob = y_prob[:, i]
        pred = y_pred[:, i]
        
        # False Positives (predicted 1, true 0)
        fp_idx = np.where((gt == 0) & (pred == 1))[0]
        fp_sorted = fp_idx[np.argsort(prob[fp_idx])[::-1]] if len(fp_idx) > 0 else []
        
        # False Negatives (predicted 0, true 1)
        fn_idx = np.where((gt == 1) & (pred == 0))[0]
        fn_sorted = fn_idx[np.argsort(prob[fn_idx])] if len(fn_idx) > 0 else []
        
        # Hard examples (max BCE loss)
        bce = -(gt * np.log(prob + 1e-8) + (1 - gt) * np.log(1 - prob + 1e-8))
        hard_sorted = np.argsort(bce)[::-1]
        
        # True Positives
        tp_idx = np.where((gt == 1) & (pred == 1))[0]
        tp_sorted = tp_idx[np.argsort(prob[tp_idx])[::-1]] if len(tp_idx) > 0 else []
        
        print(f"\nError analysis details for: {cls}")
        
        categories = [
            ("False_Positives", fp_sorted[:n]),
            ("False_Negatives", fn_sorted[:n]),
            ("Hard_Examples", hard_sorted[:n]),
            ("Top_Correct_Predictions", tp_sorted[:n])
        ]
        
        for cat_name, idx_list in categories:
            if len(idx_list) == 0:
                continue
                
            fig, axes = plt.subplots(1, len(idx_list), figsize=(4 * len(idx_list), 4))
            if len(idx_list) == 1:
                axes = [axes]
                
            for ax, idx in zip(axes, idx_list):
                img_name = df.iloc[idx]["Image Index"]
                img_path = image_dir / img_name
                
                try:
                    img = load_image_as_rgb(img_path)
                    ax.imshow(img)
                except Exception:
                    ax.fill_between([0, 1], [0, 1], color="black")
                    
                ax.set_title(f"GT: {int(gt[idx])} | Prob: {prob[idx]*100:.1f}%\n{img_name}", fontsize=8)
                ax.axis("off")
                
            plt.suptitle(f"{cls} — Error Category: {cat_name.replace('_', ' ')}", fontsize=12, fontweight="bold")
            plt.tight_layout()
            
            # Save error plot locally
            fname = f"error_{cls.lower()}_{cat_name.lower()}.png"
            plt.savefig(fname, dpi=120, bbox_inches="tight")
            plt.close()
            print(f"  Saved error plot grid to: {fname}")


## ⚡ Section 8 — Test-Time Augmentation (TTA) & Inference Pipeline
Implements the inference orchestrator supporting multi-scale image views, probability averages, and CSV exports.


In [ ]:
class TTAInference:
    """
    Averages predictions across 3 scales and horizontal flips.
    """
    def __init__(self, model: nn.Module, cfg: Config, device: torch.device):
        self.model = model.eval()
        self.cfg = cfg
        self.device = device

    @torch.no_grad()
    def predict_single(self, image_path: Union[str, Path]) -> np.ndarray:
        """Single image TTA inference. Runs 6 forward passes."""
        img = load_image_as_rgb(image_path)
        all_probs = []
        
        for scale in self.cfg.TTA_SCALES:
            tfm = build_tta_transforms(scale, self.cfg)
            
            # Normal scale view
            tensor = tfm(image=img)["image"].unsqueeze(0).to(self.device)
            with autocast(enabled=self.cfg.AMP):
                logits = self.model(tensor)
            prob_orig = torch.sigmoid(logits).cpu().numpy().squeeze()
            all_probs.append(prob_orig)
            
            # Flipped scale view
            img_flipped = cv2.flip(img, 1)
            tensor_flipped = tfm(image=img_flipped)["image"].unsqueeze(0).to(self.device)
            with autocast(enabled=self.cfg.AMP):
                logits_flipped = self.model(tensor_flipped)
            prob_flipped = torch.sigmoid(logits_flipped).cpu().numpy().squeeze()
            all_probs.append(prob_flipped)
            
        return np.mean(all_probs, axis=0)

    @torch.no_grad()
    def predict_loader(self, loader: DataLoader) -> np.ndarray:
        """Batch dataset TTA. Rebuilds loaders dynamically for target resolutions."""
        accumulated_probs = None
        count_views = 0
        
        raw_df = loader.dataset.df
        raw_img_dir = loader.dataset.image_dir
        
        for scale in self.cfg.TTA_SCALES:
            # Scale view
            tfm_norm = build_tta_transforms(scale, self.cfg)
            ds_norm = ChestXrayInferenceDataset(raw_df, raw_img_dir, tfm_norm, self.cfg)
            loader_norm = DataLoader(ds_norm, batch_size=loader.batch_size, shuffle=False,
                                     num_workers=self.cfg.NUM_WORKERS, pin_memory=self.cfg.PIN_MEMORY)
            
            probs_scale = []
            for images, _, _ in loader_norm:
                images = images.to(self.device, non_blocking=True)
                with autocast(enabled=self.cfg.AMP):
                    logits = self.model(images)
                probs_scale.append(torch.sigmoid(logits).cpu().numpy())
            probs_scale = np.concatenate(probs_scale, axis=0)
            
            if accumulated_probs is None:
                accumulated_probs = probs_scale
            else:
                accumulated_probs += probs_scale
            count_views += 1
            
            # Flip scale view
            tfm_flip = A.Compose([
                A.Resize(scale, scale, always_apply=True),
                A.HorizontalFlip(always_apply=True),
                A.Normalize(mean=self.cfg.MEAN, std=self.cfg.STD, always_apply=True),
                ToTensorV2(always_apply=True)
            ])
            ds_flip = ChestXrayInferenceDataset(raw_df, raw_img_dir, tfm_flip, self.cfg)
            loader_flip = DataLoader(ds_flip, batch_size=loader.batch_size, shuffle=False,
                                     num_workers=self.cfg.NUM_WORKERS, pin_memory=self.cfg.PIN_MEMORY)
            
            probs_flip = []
            for images, _, _ in loader_flip:
                images = images.to(self.device, non_blocking=True)
                with autocast(enabled=self.cfg.AMP):
                    logits = self.model(images)
                probs_flip.append(torch.sigmoid(logits).cpu().numpy())
            probs_flip = np.concatenate(probs_flip, axis=0)
            
            accumulated_probs += probs_flip
            count_views += 1
            
        return accumulated_probs / count_views

class InferencePipeline:
    """
    Production-grade inference interface.
    """
    def __init__(self, model: nn.Module, cfg: Config, thresholds: Optional[np.ndarray], device: torch.device):
        self.model = model.eval()
        self.cfg = cfg
        self.device = device
        self.thresholds = thresholds if thresholds is not None else np.full(cfg.NUM_CLASSES, 0.5)
        self.tta_inference = TTAInference(model, cfg, device)
        self.standard_tfm = build_transforms("val", cfg)

    def predict_single(self, image_path: Union[str, Path], use_tta: bool = True) -> Dict:
        """Runs prediction on a target image path."""
        t0 = time.time()
        if use_tta:
            probs = self.tta_inference.predict_single(image_path)
        else:
            img = load_image_as_rgb(image_path)
            tensor = self.standard_tfm(image=img)["image"].unsqueeze(0).to(self.device)
            with torch.no_grad():
                with autocast(enabled=self.cfg.AMP):
                    logits = self.model(tensor)
            probs = torch.sigmoid(logits).cpu().numpy().squeeze()
            
        elapsed = time.time() - t0
        predictions = (probs >= self.thresholds).astype(int)
        
        findings = dict(zip(self.cfg.CLASSES, probs.tolist()))
        detected = [cls for cls, pred in zip(self.cfg.CLASSES, predictions.tolist()) if pred == 1]
        
        return {
            "probabilities": findings,
            "predictions": dict(zip(self.cfg.CLASSES, predictions.tolist())),
            "detected_labels": detected if detected else ["No Finding"],
            "inference_time": elapsed
        }

    def predict_batch(self, df: pd.DataFrame, image_dir: Path, use_tta: bool = True) -> np.ndarray:
        """Runs batch predictions over a DataFrame list."""
        dataset = ChestXrayInferenceDataset(df, image_dir, self.standard_tfm, self.cfg)
        loader = DataLoader(dataset, batch_size=self.cfg.BATCH_SIZE, shuffle=False,
                            num_workers=self.cfg.NUM_WORKERS, pin_memory=self.cfg.PIN_MEMORY)
        
        if use_tta:
            probs = self.tta_inference.predict_loader(loader)
        else:
            probs = []
            with torch.no_grad():
                for images, _, _ in loader:
                    images = images.to(self.device, non_blocking=True)
                    with autocast(enabled=self.cfg.AMP):
                        logits = self.model(images)
                    probs.append(torch.sigmoid(logits).cpu().numpy())
            probs = np.concatenate(probs, axis=0)
            
        return probs


## 🚀 Section 9 — Execution Orchestration
Triggers corresponding pipeline modules depending on the configuration defined at the top selector cell.


In [ ]:
# ==============================================================================
# CLINICAI END-TO-END EXECUTION BLOCK
# ==============================================================================

# 1. Initialize configuration and rebuild model architecture from checkpoint
print("Initializing model reconstructor...")
try:
    model, checkpoint = load_checkpoint_and_rebuild(DEVICE)
except Exception as e:
    print(f"CRITICAL ERROR: Failed to load checkpoint weights: {e}")
    print("Ensure 'best_model.pth' is uploaded to kaggle working/input checkpoints folder.")
    sys.exit(1)

# 2. Check for thresholds dictionary
thresholds = np.full(CFG.NUM_CLASSES, 0.5)
if "metrics" in checkpoint and "thresholds" in checkpoint.get("metrics", {}):
    thresholds = np.array(checkpoint["metrics"]["thresholds"])
    print("Restored thresholds from checkpoint metrics.")
elif Path("thresholds.json").exists():
    with open("thresholds.json", "r") as f:
        t_dict = json.load(f)
        thresholds = np.array([t_dict.get(cls, 0.5) for cls in CFG.CLASSES])
    print("Restored thresholds from thresholds.json.")

# Instantiate Inference Pipeline
pipeline = InferencePipeline(model, CFG, thresholds, DEVICE)

# ------------------------------------------------------------------------------
# MODE: SINGLE IMAGE PREDICTION
# ------------------------------------------------------------------------------
if MODE == "single":
    print(f"\n--- Running Mode 1: Single Prediction for {IMAGE_PATH} ---")
    
    try:
        results = pipeline.predict_single(IMAGE_PATH, use_tta=USE_TTA)
    except Exception as e:
        print(f"Failed to execute prediction for {IMAGE_PATH}: {e}")
        sys.exit(1)
        
    print(f"Inference Time: {results['inference_time']:.3f} seconds")
    print("-" * 50)
    print("Most Likely Thoracic Pathologies (Sorted Descending):")
    print("-" * 50)
    
    # Sort results
    sorted_probs = sorted(results["probabilities"].items(), key=lambda x: x[1], reverse=True)
    for cls, prob in sorted_probs:
        is_pos = results["predictions"][cls] == 1
        indicator = "[X] POSITIVE" if is_pos else "[ ] NEGATIVE"
        print(f"  {indicator:<12} | {cls:<24} : {prob*100:.1f} %")
    print("-" * 50)
    print(f"Clinician Conclusion: {results['detected_labels']}")
    
    # Render beautiful visualization chart in output
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Original Patient image
    try:
        orig_img = load_image_as_rgb(IMAGE_PATH)
        axes[0].imshow(orig_img)
        axes[0].set_title(f"Patient Chest X-Ray\n{Path(IMAGE_PATH).name}", fontsize=12, fontweight="bold")
    except Exception:
        axes[0].fill_between([0,1], [0,1], color="black")
    axes[0].axis("off")
    
    # Probabilities bar chart
    cls_names = [x[0] for x in sorted_probs][::-1]
    cls_probs = [x[1] for x in sorted_probs][::-1]
    
    # Dynamic colors: red for active predictions, steelblue for negatives
    colors = []
    for cls in cls_names:
        if results["predictions"][cls] == 1:
            colors.append("#C0392B")
        else:
            colors.append("#4682B4")
            
    bars = axes[1].barh(cls_names, cls_probs, color=colors, edgecolor="gray")
    axes[1].set_xlim([0.0, 1.0])
    axes[1].set_xlabel("Predicted Probability", fontsize=11)
    axes[1].set_title("Thoracic Findings Diagnostics Probability", fontsize=12, fontweight="bold")
    
    # Add boundary indicator lines
    for idx, name in enumerate(cls_names):
        thresh_val = dict(zip(CFG.CLASSES, thresholds.tolist()))[name]
        axes[1].plot([thresh_val, thresh_val], [idx - 0.4, idx + 0.4], color="black", linestyle="--", alpha=0.6)
        
    plt.tight_layout()
    plt.savefig("single_prediction_results.png", dpi=150)
    plt.show()
    plt.close()
    
    # GradCAM overlays
    active_indices = []
    for idx, cls in enumerate(CFG.CLASSES):
        p_val = results["probabilities"][cls]
        if p_val >= GRADCAM_PROB_THRESHOLD:
            active_indices.append(idx)
            
    gradcam_paths = []
    if len(active_indices) > 0 and GRADCAM_AVAILABLE:
        print(f"\nGenerating GradCAM Heatmaps for diseases exceeding explainability threshold ({GRADCAM_PROB_THRESHOLD})...")
        gradcam_paths = generate_gradcam_visualizations(model, IMAGE_PATH, active_indices, CFG, DEVICE, show_plot=True)
        
    # PDF Clinical report card builder
    primary_cam = gradcam_paths[0] if len(gradcam_paths) > 0 else None
    
    # Compile report
    print("\nCompiling Clinical PDF Diagnostic report card...")
    generate_pdf_report(
        output_path="report.pdf",
        patient_img_path=IMAGE_PATH,
        gradcam_img_path=primary_cam,
        predictions=results["probabilities"],
        thresholds=dict(zip(CFG.CLASSES, thresholds.tolist())),
        top_diagnosis=sorted_probs[0][0],
        top_prob=sorted_probs[0][1],
        model_name=CFG.BACKBONE,
        inference_time=results["inference_time"],
        device_name=str(DEVICE)
    )

# ------------------------------------------------------------------------------
# MODE: BATCH PREDICTION
# ------------------------------------------------------------------------------
elif MODE == "batch":
    print(f"\n--- Running Mode 3: Batch Inference for {BATCH_INPUT_FOLDER} ---")
    batch_dir = Path(BATCH_INPUT_FOLDER)
    if not batch_dir.exists():
        print(f"CRITICAL ERROR: Input folder '{BATCH_INPUT_FOLDER}' does not exist.")
        sys.exit(1)
        
    # Filter files
    valid_exts = {".png", ".jpeg", ".jpg", ".dcm"}
    img_files = [f for f in batch_dir.iterdir() if f.suffix.lower() in valid_exts]
    print(f"Found {len(img_files)} images in directory.")
    
    if len(img_files) == 0:
        print("No compatible medical images found.")
        sys.exit(1)
        
    if BATCH_LIMIT is not None:
        img_files = img_files[:BATCH_LIMIT]
        print(f"Batch execution limited to first {BATCH_LIMIT} images.")
        
    # Build batch dataframe
    batch_df = pd.DataFrame([f.name for f in img_files], columns=["Image Index"])
    
    # Predict probabilities
    t0 = time.time()
    probs = pipeline.predict_batch(batch_df, batch_dir, use_tta=USE_TTA)
    elapsed = time.time() - t0
    
    print(f"Batch prediction finished in {elapsed:.2f} seconds.")
    
    # Rebuild prediction dataframes
    preds = (probs >= thresholds).astype(int)
    
    # Add details
    result_df = pd.DataFrame(probs, columns=CFG.CLASSES)
    result_df.insert(0, "filename", batch_df["Image Index"])
    
    # Determine top disease
    top_indices = np.argmax(probs, axis=1)
    result_df["top disease"] = [CFG.CLASSES[idx] for idx in top_indices]
    result_df["top probability"] = np.max(probs, axis=1)
    
    result_df.to_csv("predictions.csv", index=False)
    print("Exported batch predictions to: predictions.csv")
    print(result_df.head(10))

# ------------------------------------------------------------------------------
# MODE: TEST SET EVALUATION
# ------------------------------------------------------------------------------
elif MODE == "dataset":
    print("\n--- Running Mode 4: Test set evaluation ---")
    
    # Load test split file if available
    test_list_file = CFG.DATA_DIR / "test_list.txt"
    if not test_list_file.exists():
        test_list_file = Path("test_list.txt")
        
    df_raw = pd.read_csv(CFG.META_CSV)
    
    if test_list_file.exists():
        print(f"Reading test split filenames from: {test_list_file}")
        with open(test_list_file, "r") as f:
            test_images = {line.strip() for line in f if line.strip()}
        eval_df = df_raw[df_raw["Image Index"].isin(test_images)].reset_index(drop=True)
        print(f"Filtered metadata CSV: {len(eval_df)} images found for evaluation.")
    else:
        print("WARNING: test_list.txt not found. Splitting val set patient-wise from metadata CSV.")
        # Perform patient-wise train/test split fallback
        patient_ids = df_raw["Patient ID"].apply(lambda x: int(str(x).split("_")[0]) if "_" in str(x) else int(x)).unique()
        _, test_patients = train_test_split(patient_ids, test_size=0.10, random_state=CFG.SEED)
        df_raw["Patient ID Parsed"] = df_raw["Patient ID"].apply(lambda x: int(str(x).split("_")[0]) if "_" in str(x) else int(x))
        eval_df = df_raw[df_raw["Patient ID Parsed"].isin(test_patients)].reset_index(drop=True)
        print(f"Split completed: evaluation dataframe size = {len(eval_df)} images.")
        
    # Map multi-label binary labels
    for cls in CFG.CLASSES:
        eval_df[cls] = eval_df["Finding Labels"].apply(lambda x: 1 if cls in str(x).split("|") else 0)
        
    # We slice evaluation dataset to speed up notebook runs on CPU or Kaggle tests
    if len(eval_df) > 500:
        eval_df = eval_df.sample(n=500, random_state=CFG.SEED).reset_index(drop=True)
        print(f"Sliced evaluation data to 500 samples for verification run times.")
        
    # Run predictions
    print(f"Evaluating predictions for {len(eval_df)} images...")
    t0 = time.time()
    eval_probs = pipeline.predict_batch(eval_df, CFG.IMAGE_DIR, use_tta=USE_TTA)
    eval_time = time.time() - t0
    print(f"Inference complete in {eval_time/60:.2f} minutes.")
    
    eval_labels = eval_df[CFG.CLASSES].values.astype(np.float32)
    
    # Run Threshold Optimization (Mode 6)
    optimized_thresholds = run_threshold_optimization(eval_labels, eval_probs)
    
    # Re-evaluate metrics with optimized boundaries
    metrics_default = compute_evaluation_metrics(eval_labels, eval_probs, np.full(CFG.NUM_CLASSES, 0.5))
    metrics_opt = compute_evaluation_metrics(eval_labels, eval_probs, optimized_thresholds)
    
    print(f"\n{'Metric':<22} | {'Default (0.5)':<16} | {'Optimized':<16}")
    print("-" * 60)
    for m in ["auc_macro", "auc_micro", "ap_macro", "f1_macro", "precision_macro", "recall_macro", "specificity_macro"]:
        print(f"{m:<22} | {metrics_default[m]:<16.4f} | {metrics_opt[m]:<16.4f}")
        
    # Export metrics CSV
    rows = []
    for cls in CFG.CLASSES:
        cls_m_def = metrics_default["per_class"][cls]
        cls_m_opt = metrics_opt["per_class"][cls]
        rows.append({
            "Pathology": cls,
            "AUC": cls_m_opt["auc"],
            "AP": cls_m_opt["ap"],
            "Default Threshold": 0.5,
            "Default F1": cls_m_def["f1"],
            "Optimized Threshold": dict(zip(CFG.CLASSES, optimized_thresholds.tolist()))[cls],
            "Optimized F1": cls_m_opt["f1"],
            "Precision": cls_m_opt["precision"],
            "Recall": cls_m_opt["recall"],
            "Specificity": cls_m_opt["specificity"]
        })
        
    metrics_df = pd.DataFrame(rows)
    metrics_df.to_csv("metrics.csv", index=False)
    print("\nExported metrics table to: metrics.csv")
    
    # Generate Evaluation plots (Mode 5)
    plot_performance_graphs(eval_labels, eval_probs, optimized_thresholds, CFG.OUTPUT_DIR)
    
    # Error Analysis (Mode 7)
    run_error_analysis(eval_df, eval_labels, eval_probs, optimized_thresholds, CFG.IMAGE_DIR, n=4)

print("\nClinicAI Inference Execution Complete!")
